# 가설 2 | GTR 20,000에피소드 공통 학습·평가

**본 노트북은 20,000에피소드 최종 학습용입니다.** 5,000에피소드 실험은 Buffer 30,000 / Epsilon decay 500,000을 선정하기 위한 사전 실험이며, 기존 5,000판 가중치에서 학습을 재개하지 않습니다.

- 공통 모델: GTR afterstate value network, 1-step TD, 1-ply 학습
- 학습: 20,000에피소드, Buffer 30,000, Epsilon decay 500,000, 시드 12345
- 중간 검증: 매 500에피소드, 1-ply Greedy 100판(시드 200000~200099)
- 모델 선택: 평균 최대 연쇄 우선, 동률이면 평균 점수. `best_chain.pt` 자동 보존
- 최종 평가: 선택 모델에 대해 별도 3-ply 250판(시드 100000~100249)
- 후속: 민규님이 동일한 20k 모델로 B/C 탐색을 구현하고 A/B/C 평가 데이터를 비교합니다.

**Kaggle에서 GPU를 설정하고 순서대로 실행해 주십시오.** 최종 3-ply 평가는 매우 오래 걸릴 수 있어 기본값을 꺼 두었습니다. 20k 본 학습은 처음부터 새로 수행하도록 설정되어 있습니다.


In [ ]:
N_STEP = 1
TOTAL_EPISODES = 20000
DEBUG_QUICK = False
DO_TRAIN = True
RUN_FINAL_EVALUATION = False
RESUME_CHECKPOINT = None
EVAL_CHECKPOINT = None
COMPARE_RUNS = []
HORIZON_A_EPISODES = None
HORIZON_B_EPISODES = None
HORIZON_C_EPISODES = None


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import csv
import hashlib
import json
import importlib.util
import os
import re
import subprocess
import sys
import sysconfig
import zipfile
import torch

assert N_STEP in (1, 3, 5)
assert TOTAL_EPISODES == 20000 or DEBUG_QUICK
kaggle_root = Path('/kaggle/working')
if not kaggle_root.is_dir():
    raise RuntimeError('Kaggle 환경에서 실행해 주십시오.')
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU 가속기를 활성화해 주십시오.')
print('GPU:', torch.cuda.get_device_name(0))
project = kaggle_root / 'puyo_gtr_h2_20k_project'
project.mkdir(parents=True, exist_ok=True)
runs = kaggle_root / 'puyo_runs' / 'gtr_horizon2_20k'
runs.mkdir(parents=True, exist_ok=True)


In [ ]:
EMBEDDED_ARCHIVE_BASE64 = (
    '...'
    'AAAAGQAZAF0GAACsrgAAAAA='
)
archive_path = kaggle_root / 'gtr_horizon2_20k_source.zip'
archive_path.write_bytes(base64.b64decode(EMBEDDED_ARCHIVE_BASE64))
with zipfile.ZipFile(archive_path) as package:
    package.extractall(project)
print('통합 소스:', project)


In [ ]:
                                       
def run_cmd(args, cwd=project):
    cmd=[str(x) for x in args]
    print('실행:', ' '.join(cmd[:8]), ('...' if len(cmd)>8 else ''), flush=True)
    with subprocess.Popen(cmd, cwd=str(cwd), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
        rc=proc.wait()
    if rc:
        raise subprocess.CalledProcessError(rc,cmd)

if importlib.util.find_spec('pybind11') is None:
    print('pybind11 설치 중... Kaggle Internet 옵션이 꺼져 있으면 실패할 수 있습니다.')
    run_cmd([sys.executable,'-m','pip','install','-q','pybind11==3.1.0'])
import pybind11
ext_path=project / ('rl_env/puyo_cpp'+sysconfig.get_config_var('EXT_SUFFIX'))
if not ext_path.exists():
    run_cmd(['g++','-std=c++20','-O2','-march=native','-DPEXT','-shared','-fPIC',
             '-I'+pybind11.get_include(),'-I'+sysconfig.get_path('include'),
             *sorted(str(f) for f in (project/'core').glob('*.cpp')),
             str(project/'rl_env/bindings.cpp'),'-o',str(ext_path)])
run_cmd([sys.executable,'-c',
         'from rl_env import puyo_cpp; e=puyo_cpp.PuyoEnv(); e.reset(12345); assert e.clone().preview_all(); print("게임 엔진 정상")'])

In [ ]:
team = json.loads((project / 'team_20k_config.json').read_text(encoding='utf-8'))
cfg = dict(team)
cfg['n_step'] = N_STEP
if DEBUG_QUICK:
    cfg.update(episodes=12,channels=8,blocks=1,batch_size=4,capacity=64,
               warmup=4,eval_every=6,eval_episodes=2,save_every=6,resume_every=6)
mode = 'DEBUG' if DEBUG_QUICK else 'MAIN'
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
destination = runs / f'{mode}_H2_B{cfg["capacity"]}_E{cfg["epsilon_steps"]}_n{N_STEP}_{stamp}'
resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
if resume is not None and not resume.is_file():
    raise FileNotFoundError(resume)
args = [
    sys.executable,'-u','puyo_train.py','--device','cuda',
    '--episodes',str(cfg['episodes']),'--n-step',str(cfg['n_step']),
    '--seed',str(cfg['seed']),'--train-ply',str(cfg['train_ply']),
    '--eval-ply',str(cfg['eval_ply']),'--eval-every',str(cfg['eval_every']),
    '--eval-episodes',str(cfg['eval_episodes']),'--eval-seed',str(cfg['eval_seed']),
    '--save-every',str(cfg['save_every']),'--resume-every',str(cfg['resume_every']),
    '--lr',str(cfg['lr']),'--batch-size',str(cfg['batch_size']),
    '--gamma',str(cfg['gamma']),'--form-weight',str(cfg['form_weight']),
    '--capacity',str(cfg['capacity']),'--warmup',str(cfg['warmup']),
    '--update-every',str(cfg['update_every']),'--target-sync',str(cfg['target_sync']),
    '--epsilon-start',str(cfg['epsilon_start']),
    '--epsilon-final',str(cfg['epsilon_final']),
    '--epsilon-steps',str(cfg['epsilon_steps']),
    '--per-alpha',str(cfg['per_alpha']),
    '--per-beta-start',str(cfg['per_beta_start']),
    '--per-beta-steps',str(cfg['per_beta_steps']),
    '--per-mode',str(cfg['per_mode']),
    '--score-scale',str(cfg['score_scale']),
    '--gameover-penalty',str(cfg['gameover_penalty']),
    '--channels',str(cfg['channels']),'--blocks',str(cfg['blocks']),
    '--eval-root-keep',str(cfg['eval_root_keep']),
    '--eval-second-keep',str(cfg['eval_second_keep']),
    '--no-eval-moves','--run-dir',str(destination),
]
if DEBUG_QUICK:
    args += ['--no-probes','--no-train-moves','--print-every','6']
if resume is not None:
    args += ['--resume',str(resume)]
print('학습 목표:',cfg['episodes'],'episodes')
print('모델:',cfg['channels'],'channels',cfg['blocks'],'blocks')
print('선정 파라미터:',cfg['capacity'],cfg['epsilon_steps'])
print('Greedy 중간 검증:',cfg['eval_every'],cfg['eval_episodes'],cfg['eval_seed'])
print('학습 출력:', destination)


In [ ]:
if DO_TRAIN:
    run_cmd(args)
    completed = destination / 'completed.json'
    if not completed.is_file():
        raise RuntimeError('학습 완료 기록이 생성되지 않았습니다.')
    status = json.loads(completed.read_text(encoding='utf-8'))
    if int(status.get('end_episode', -1)) != cfg['episodes']:
        raise RuntimeError('학습 에피소드 수가 목표와 일치하지 않습니다.')
    for f in ['manifest.json','train_episodes.csv','eval_metrics.csv',
              'best_chain.pt','best_chain_selection.json','last_resume.pt',
              f'ep_{cfg["episodes"]:06d}.pt']:
        q = destination / f
        if not q.is_file():
            raise FileNotFoundError(q)
    if int(status.get('gradient_updates',0)) < 1:
        raise RuntimeError('신경망 업데이트가 확인되지 않았습니다.')
    print('학습 및 검증 완료:',destination)
else:
    print('학습은 수행하지 않았습니다.')


In [ ]:
if DO_TRAIN and (destination / 'eval_metrics.csv').is_file():
    import matplotlib.pyplot as plt
    with (destination / 'eval_metrics.csv').open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    episodes = [int(r['episode']) for r in rows]
    chains = [float(r['mean_max_chain']) for r in rows]
    scores = [float(r['mean_score']) for r in rows]
    fig, ax = plt.subplots(figsize=(8,4.2))
    ax.plot(episodes,chains,marker='o',markersize=3)
    ax.set_xlabel('Training episode')
    ax.set_ylabel('1-ply greedy mean max chain')
    ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(destination / 'greedy_validation_curve.png',dpi=180)
    plt.close(fig)
    result = json.loads((destination / 'best_chain_selection.json').read_text(encoding='utf-8'))
    print('최고 모델:',destination / 'best_chain.pt')
    print('선정 에피소드:',result['episode'],'평균 최대 연쇄:',result['mean_max_chain'])


In [ ]:
if RUN_FINAL_EVALUATION:
    checkpoint = Path(EVAL_CHECKPOINT) if EVAL_CHECKPOINT else destination / 'best_chain.pt'
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    out = runs / 'evaluations' / f'{checkpoint.parent.name}_{checkpoint.stem}_ply3_{timestamp}'
    run_cmd([sys.executable,'-u','puyo_train.py','evaluate',
             '--checkpoint',str(checkpoint),'--episodes','250',
             '--seed','100000','--ply','3','--root-keep','22',
             '--second-keep','484','--device','cuda',
             '--out-dir',str(out),'--summary-only'])
    summary = json.loads((out / 'summary.json').read_text(encoding='utf-8'))
    provenance = dict(variant='A_short_visible_3ply',
                      backbone_checkpoint_path=str(checkpoint),checkpoint_sha256=digest,
                      backbone_td_n_step=N_STEP,visible_pairs=3,evaluation_ply=3,
                      evaluation_games=250,evaluation_seed_start=100000,
                      status='ACTUAL_A_EVALUATION')
    (out / 'horizon2_A_provenance.json').write_text(
        json.dumps(provenance,ensure_ascii=False,indent=2),encoding='utf-8')
    print('3-ply 평가:',summary['mean_max_chain'],summary['mean_score'])
    print('결과 폴더:',out)
else:
    print('3-ply 최종 평가는 별도로 실행해 주십시오.')


In [ ]:
if HORIZON_A_EPISODES:
    folder = runs / ('horizon2_comparison_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
    cmd = [sys.executable,'analysis/horizon2_compare.py','--a',str(HORIZON_A_EPISODES),'--out',str(folder)]
    if HORIZON_B_EPISODES: cmd += ['--b',str(HORIZON_B_EPISODES)]
    if HORIZON_C_EPISODES: cmd += ['--c',str(HORIZON_C_EPISODES)]
    run_cmd(cmd)
    print('A/B/C 분석:',folder)
else:
    print('A/B/C 평가 자료의 경로를 설정하면 비교가 가능합니다.')


## 학습 재개 및 후속 인계

- **기본 실행은 0→20,000에피소드 신규 학습입니다.** 5,000판 평가용 `ep_005000.pt`에서 이어서 학습하지 않습니다.
- 중단된 20,000판 학습을 재개할 때는 보존된 `last_resume.pt`의 경로를 `RESUME_CHECKPOINT`에 지정한 뒤, 나머지 학습 설정과 총 에피소드 목표를 동일하게 유지해 주십시오.
- Kaggle 세션 종료 전에 Output을 보존해 주십시오. `last_resume.pt`에는 Optimizer·Replay·난수 상태가 포함됩니다.
- `best_chain.pt`는 중간 1-ply 검증에서 선별된 평가용 모델입니다. 이후 A/B/C는 **동일한 20k `best_chain.pt` 가중치**를 공통으로 사용해 주십시오.
- 3-ply 250판 평가에 필요한 시간은 GPU·환경에 따라 상당히 길 수 있습니다. 별도 실행 예산을 확보하여 진행해 주십시오.
